# 🌸 Orchid 1.0 — Primer LLM Colombiano

**First competitive LLM trained and aligned in Colombia** — 2B ternary-weight model built on [BitNet b1.58](https://huggingface.co/microsoft/bitnet-b1.58-2B-4T), aligned with ORPO, served by [ternative](https://github.com/michelangeloromerochisco/ternative).

Model: [MicheRomChis/orchid-1.0](https://huggingface.co/MicheRomChis/orchid-1.0) · License: Apache 2.0

---

### Instructions

1. **Runtime → Change runtime type → T4 GPU** (recommended, free) — or leave on CPU
2. **Runtime → Run all** (`Ctrl+F9`)
3. Wait for the server to start:
   - **First run (GPU)**: ~5–8 min (LoRA merge + CUDA build)
   - **First run (CPU)**: ~10–12 min
   - **Subsequent runs** (same session): ~30 s
4. Click the **public Gradio URL** at the bottom

> **GPU mode**: ~10–14 tok/s on T4. **CPU mode**: ~6 tok/s with AVX2.
> The public URL stays valid for 72 hours while the session is active.

In [ ]:
# ── Cell 1: Check hardware ────────────────────────────────────
import os

os.system('echo "CPU:" && grep "model name" /proc/cpuinfo | head -1 | cut -d: -f2')
os.system('echo "Cores:" && nproc')
os.system('echo "RAM:" && free -h | awk \'/Mem:/{print $2}\'')

HAS_GPU = os.path.exists('/proc/driver/nvidia/version')

if HAS_GPU:
    os.system('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader')
    print('\nGPU: ✓ CUDA available — will build GPU backend (~10–14 tok/s)')
else:
    flags = open('/proc/cpuinfo').read()
    avx2  = 'avx2' in flags
    print(f'\nGPU: not available — CPU mode (~6 tok/s {"with AVX2" if avx2 else "WARNING: no AVX2, will be slower"})')

In [ ]:
# ── Cell 2: Clone ternative ───────────────────────────────────
import os, sys

os.system('rm -rf /content/ternative')

print('Cloning ternative...')
ret = os.system('git clone --depth 1 https://github.com/michelangeloromerochisco/ternative /content/ternative')
if ret != 0:
    print('Clone FAILED'); sys.exit(1)

for f in ['CMakeLists.txt', 'src/model.cpp', 'cuda/ops_cuda.cu']:
    path = f'/content/ternative/{f}'
    ok   = os.path.exists(path)
    print(f'{"OK" if ok else "MISSING"}  {f}')
    if not ok:
        sys.exit(1)

print('Source ready.')

In [ ]:
# ── Cell 3: Build ternative (GPU if available, CPU fallback) ──
import subprocess, sys, os

HAS_GPU   = os.path.exists('/proc/driver/nvidia/version')
CUDA_FLAG = 'ON' if HAS_GPU else 'OFF'
print(f'CUDA={CUDA_FLAG} | Building ternative...')

subprocess.run(['apt-get', 'install', '-y', '-q',
                'cmake', 'libopenblas-dev', 'libgomp1'], check=True)

os.system('rm -rf /content/ternative/build')

r = subprocess.run(
    ['cmake', '-B', 'build',
     '-DCMAKE_BUILD_TYPE=Release',
     f'-DTERNATIVE_CUDA={CUDA_FLAG}',
     '-DTERNATIVE_BUILD_TESTS=OFF'],
    cwd='/content/ternative',
    capture_output=True, text=True
)
if r.returncode != 0:
    print('cmake FAILED:\n', r.stderr[-2000:]); sys.exit(1)
print('cmake OK. Compiling...')

r = subprocess.run(
    ['cmake', '--build', 'build', '--parallel', '4'],
    cwd='/content/ternative',
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Build FAILED:\n', r.stderr[-3000:]); sys.exit(1)

os.system('cp /content/ternative/build/ternative /usr/local/bin/ternative')
mode = 'GPU — ~10–14 tok/s' if HAS_GPU else 'CPU — ~6 tok/s'
print(f'Build complete! Inference mode: {mode}')

In [ ]:
# ── Cell 4: Download Orchid models ───────────────────────────
from huggingface_hub import hf_hub_download
import os

os.makedirs('/content/models', exist_ok=True)
MODEL_REPO = 'MicheRomChis/orchid-1.0'

print('Downloading base model (~1.1 GB)...')
hf_hub_download(MODEL_REPO, 'ggml-model-i2_s.gguf',  local_dir='/content/models')
print('Downloading LoRA adapter (~90 MB)...')
hf_hub_download(MODEL_REPO, 'dpo_aligned-lora.gguf', local_dir='/content/models')
print('Models ready!')
os.system('ls -lh /content/models/')

In [ ]:
# ── Cell 5: Start ternative server ───────────────────────────
import subprocess, threading, time, requests as req, os, sys

PORT      = 8899
CACHE_DIR = '/content/.ternative_cache'
HAS_GPU   = os.path.exists('/proc/driver/nvidia/version')

os.makedirs(CACHE_DIR, exist_ok=True)
os.system(f'fuser -k {PORT}/tcp 2>/dev/null; pkill -9 -f ternative 2>/dev/null; sleep 2')

env = os.environ.copy()
env['TERNATIVE_CACHE_DIR'] = CACHE_DIR

cmd = ['ternative',
       '--model', '/content/models/ggml-model-i2_s.gguf',
       '--lora',  '/content/models/dpo_aligned-lora.gguf',
       '--server', '--port', str(PORT)]
if not HAS_GPU:
    cmd.append('--no-gpu')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)

def _log():
    for line in iter(proc.stdout.readline, ''):
        l = line.rstrip()
        if l and '[apply_lora]' not in l:
            print('[ternative]', l)

threading.Thread(target=_log, daemon=True).start()

mode = 'GPU' if HAS_GPU else 'CPU'
print(f'Starting ternative ({mode} mode) on port {PORT}...')
print('First run: LoRA merge in progress (5–10 min). Subsequent runs: ~30 s.\n')

for i in range(600):
    time.sleep(1)
    if proc.poll() is not None:
        print(f'ERROR: ternative exited (code {proc.returncode})'); sys.exit(1)
    try:
        r = req.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=2)
        if r.status_code == 200:
            print(f'\nServer ready after {i+1}s ✓  ({mode} inference)')
            break
    except Exception:
        pass
    if (i+1) % 60 == 0:
        print(f'  {(i+1)//60} min — still loading...')
else:
    print('Timeout. Try Runtime → Restart and re-run.'); sys.exit(1)

In [ ]:
# ── Cell 6: Gradio chat UI (same as production) ───────────────
!pip install -q "gradio>=5.0,<6.0"

import gradio as gr
import requests
import os

PORT = 8899

SYSTEM_PROMPT = (
    "You are Orchid, a multilingual AI assistant built in Colombia by "
    "Michelangelo Romero Chisco. You reason carefully — for complex "
    "problems, think step by step before your final answer. "
    "Be truthful, unbiased, and concise. "
    "Reply in the language the user writes in."
)

def respond(message, history, temperature, max_tokens):
    if not message:
        return "", history or []

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for h in (history or []):
        if isinstance(h, (list, tuple)) and len(h) == 2:
            messages.append({"role": "user",      "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    timeout = max(120, max_tokens * 2)

    try:
        r = requests.post(
            f"http://127.0.0.1:{PORT}/v1/chat/completions",
            json={
                "model":       "orchid",
                "messages":    messages,
                "max_tokens":  max_tokens,
                "temperature": temperature,
                "stream":      False,
            },
            timeout=timeout,
        )
        if r.status_code == 200:
            reply = r.json()["choices"][0]["message"]["content"]
        else:
            reply = f"❌ HTTP {r.status_code}: {r.text[:300]}"
    except requests.exceptions.Timeout:
        reply = f"⏳ Timed out after {timeout}s — try fewer max tokens."
    except Exception as e:
        reply = f"❌ {e}"

    return "", (history or []) + [[message, reply]]


CSS = """
body { background: #080818 !important; }
.gradio-container { background: #080818 !important; color: #e8e8ff !important; }
.tricolor { display:flex; height:5px; width:100%; }
.tricolor span { flex:1; }
"""

HAS_GPU = os.path.exists('/proc/driver/nvidia/version')
mode_str = f"GPU T4 — ~10–14 tok/s" if HAS_GPU else "CPU — ~6 tok/s"

with gr.Blocks(title="Orchid 1.0", css=CSS) as demo:

    gr.HTML(
        '<div class="tricolor">'
        + '<span style="background:#FCD116"></span>' * 3
        + '<span style="background:#003893"></span>' * 3
        + '<span style="background:#CE1126"></span>' * 3
        + '</div>'
    )

    gr.Markdown(f"""
# 🌸 Orchid 1.0 — Primer LLM Colombiano
**2B ternary · ORPO aligned · Apache 2.0** &nbsp;|&nbsp; *{mode_str}*
""")

    chatbot = gr.Chatbot(height=480, type="tuples", label="Orchid")

    with gr.Row():
        msg      = gr.Textbox(placeholder="Hola Orchid! / Hello Orchid!", label="", scale=5)
        send_btn = gr.Button("Send ▶", scale=1, variant="primary")

    with gr.Row():
        temp      = gr.Slider(0.0, 1.5, value=0.7,  step=0.05, label="Temperature")
        max_toks  = gr.Slider(64,  512, value=256,  step=32,   label="Max tokens")
        clear_btn = gr.Button("Clear")

    gr.Markdown(
        "*[MicheRomChis/orchid-1.0](https://huggingface.co/MicheRomChis/orchid-1.0) · "
        "[ternative](https://github.com/michelangeloromerochisco/ternative) · "
        "Colombia 🇨🇴*"
    )

    msg.submit(respond,      [msg, chatbot, temp, max_toks], [msg, chatbot])
    send_btn.click(respond,  [msg, chatbot, temp, max_toks], [msg, chatbot])
    clear_btn.click(lambda: (None,), None, chatbot, queue=False)

demo.launch(share=True)